# 🎙️➡️🎨 From Voice to Vision — Fase 6: Explainability (XAI)

Apriamo la "scatola nera". Tre tecniche complementari, tutte argomenti del corso
(`11 Explainability`, `Explainable AI slide`):
- **Grad-CAM** — *dove* guarda la CNN sullo spettrogramma per decidere ogni emozione
- **t-SNE** — come sono separate (o confuse) le emozioni nello spazio degli embedding
- **SHAP** — quali feature MFCC pesano di più nel modello classico

Questo spiega anche *perché sad e fearful* sono difficili.

In [ ]:
# === 1. Setup + modello ottimizzato ===
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm shap

import numpy as np, torch, matplotlib.pyplot as plt
from src import config, data_loader, features
from src.models import cnn
data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)
data["split"] = df["split"].to_numpy()
splits = features.split_arrays(data)

device = cnn.get_device()
model, ck = cnn.load_checkpoint(config.RESULTS_DIR/"best_cnn.pt", device)
mean, std = ck["norm"]
loader = cnn.eval_loader(splits, "test", mean, std, deltas=True, batch_size=32)
acc, preds, tgts = cnn.evaluate(model, loader, device)
print(f"Modello caricato ({ck['hp']}) — test acc = {acc:.3f}")

## 1) Grad-CAM — dove guarda la rete
Sovrapponiamo la mappa Grad-CAM (in colore caldo) allo spettrogramma log-Mel, per un esempio
ben classificato di ciascuna emozione. Le zone accese sono quelle che hanno spinto la decisione.

In [ ]:
from src.explainability.gradcam import GradCAM, last_conv_block
from src.models.cnn import MelDataset
test_ds = MelDataset(splits["test"]["X_mel"], splits["test"]["y"], mean, std, deltas=True, augment=False)
cam_engine = GradCAM(model, last_conv_block(model))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, emo in enumerate(config.EMOTIONS):
    ax = axes[i//4, i%4]
    cls = config.EMOTION_TO_ID[emo]
    ok = np.where((tgts == cls) & (preds == tgts))[0]
    j = (ok if len(ok) else np.where(tgts == cls)[0])[0]
    x, _ = test_ds[j]; x = x.unsqueeze(0).to(device)
    cam, _ = cam_engine(x, class_idx=cls)
    ax.imshow(test_ds.X[j, 0], origin="lower", aspect="auto", cmap="magma")
    ax.imshow(cam, origin="lower", aspect="auto", cmap="jet", alpha=0.45)
    ax.set_title(emo); ax.set_xlabel("tempo"); ax.set_ylabel("mel"); ax.label_outer()
plt.suptitle("Grad-CAM — regioni tempo-frequenza che guidano la decisione", fontsize=14)
plt.tight_layout(); config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR/"06_gradcam.png", dpi=120); plt.show()

## 2) t-SNE degli embedding
Proiezione 2-D delle feature apprese dalla CNN sul test set. Cluster ben separati = classi facili;
cluster sovrapposti = classi che il modello confonde.

In [ ]:
from src.explainability import tsne_shap
emb = tsne_shap.extract_embeddings(model, splits["test"]["X_mel"], splits["test"]["y"], mean, std)
proj = tsne_shap.run_tsne(emb)
plt.figure(figsize=(8.5,7))
for i, emo in enumerate(config.EMOTIONS):
    m = splits["test"]["y"] == i
    plt.scatter(proj[m,0], proj[m,1], label=emo, s=22, alpha=.8)
plt.legend(loc="best"); plt.title("t-SNE degli embedding CNN (test)")
plt.xlabel("dim 1"); plt.ylabel("dim 2"); plt.tight_layout()
plt.savefig(config.FIGURES_DIR/"06_tsne.png", dpi=120); plt.show()

## 3) SHAP — importanza delle feature (modello classico)
Sul Random Forest applicato al vettore MFCC: quali coefficienti contano di più mediamente.

In [ ]:
shap_values, Xte_s, names, rf = tsne_shap.shap_random_forest(data)

# importanza globale = media di |SHAP| su classi e campioni (robusta a versioni diverse di shap)
sv = shap_values
if isinstance(sv, list):
    imp = np.mean([np.abs(s).mean(0) for s in sv], axis=0)
else:
    arr = np.abs(np.asarray(sv))
    imp = arr.mean(axis=(0,2)) if arr.ndim == 3 else arr.mean(0)

order = np.argsort(imp)[::-1][:15]
plt.figure(figsize=(9,5))
plt.barh([names[k] for k in order][::-1], imp[order][::-1], color="#4C78A8")
plt.xlabel("importanza media |SHAP|"); plt.title("Top-15 feature MFCC (Random Forest)")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/"06_shap.png", dpi=120); plt.show()
print("Feature più importanti:", [names[k] for k in order[:8]])

### ✅ Fase 6 completata
Abbiamo spiegato il modello da tre angolazioni: *dove* guarda (Grad-CAM), *come* separa le classi
(t-SNE), *quali* feature contano (SHAP).

**Mandami:** le tre figure. Nel report le commenteremo (es. perché sad/fearful si sovrappongono).
Poi la **Fase 7: l'arte** — l'emozione predetta genera un'immagine con Stable Diffusion. 🎨